# HHL Algorithm: A Toy Linear System

## Context & Motivation

HHL (Harrow-Hassidim-Lloyd) solves linear systems $A\vec{x} = \vec{b}$ for the solution vector $\vec{x}$, with a runtime that scales exponentially better than classical methods *under the right conditions* (sparse, well-conditioned $A$; you only need to extract certain properties of $\vec{x}$, not read out every entry classically). It's the algorithm behind the "differential equation solvers, data fitting" applications from Lecture 2's algorithm table.

A full HHL implementation involves Quantum Phase Estimation (QPE), a controlled rotation, and uncomputing the QPE. That's real machinery. This exercise uses a deliberately simple $2\times2$ matrix whose eigenvalues are exact small integers, so **QPE with just 2 qubits gives an exact answer with no approximation error**, letting you see every piece of HHL work correctly without the complexity of a general-purpose implementation.

## Problem Statement

We want to solve $A\vec{x} = \vec{b}$ for:
$$A = \begin{pmatrix} 1.5 & -0.5 \\ -0.5 & 1.5 \end{pmatrix}, \qquad \vec{b} = \begin{pmatrix} 0 \\ 1 \end{pmatrix} = |1\rangle$$

$A$ has eigenvalues **1** and **2**, with eigenvectors $|-\rangle$ and $|+\rangle$ respectively, which is convenient, because with the right choice of evolution time $t$, Quantum Phase Estimation can represent these exactly as 2-bit binary fractions (no approximation needed).

### Eigenvalues and eigenvectors proof

The eigenvalues satisfy

$$
\det(A-\lambda I)=0.
$$

For

$$
A=
\begin{pmatrix}
1.5 & -0.5\\
-0.5 & 1.5
\end{pmatrix},
$$

we have

$$
\det
\begin{pmatrix}
1.5-\lambda & -0.5\\
-0.5 & 1.5-\lambda
\end{pmatrix}
=(1.5-\lambda)^2-0.25=0.
$$

Therefore,

$$
1.5-\lambda=\pm0.5,
$$

giving

$$
\boxed{\lambda_1=1,\qquad \lambda_2=2}.
$$

The corresponding eigenvectors are

$$
\boxed{\lambda=1:\ |+\rangle=\frac{|0\rangle+|1\rangle}{\sqrt2}},
\qquad
\boxed{\lambda=2:\ |-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}}.
$$

For example, $A|+\rangle=|+\rangle$ and $A|-\rangle=2|-\rangle$.


## Circuit layout

- 1 **system** qubit: holds $|b\rangle$, and at the end holds (an unnormalized copy of) $\vec{x}$
- 2 **clock** qubits: used for phase estimation of $A$'s eigenvalues
- 1 **ancilla** qubit: postselecting on this being $|1\rangle$ is what encodes the $1/\lambda$ weighting into the final state

## Your Tasks

We've picked $t=\pi/2$ so that eigenvalue $\lambda=1$ maps to clock register value `01`, and $\lambda=2$ maps to `10`, exactly, no approximation. The reason is that QPE estimates the phase $\phi$ defined by

$$
U|\lambda\rangle=e^{2\pi i\phi}|\lambda\rangle.
$$

For $U=e^{iAt}$, an eigenvalue $\lambda$ therefore corresponds to

$$
\phi=\frac{\lambda t}{2\pi}.
$$

With $t=\pi/2$:

$$
\lambda=1 \quad\Rightarrow\quad \phi=\frac14=0.01_2 \quad\Rightarrow\quad |01\rangle,
$$

$$
\lambda=2 \quad\Rightarrow\quad \phi=\frac12=0.10_2 \quad\Rightarrow\quad |10\rangle.
$$

Both phases have exact two-bit binary representations, so QPE introduces no approximation error in this example.




1. Build $U = e^{iAt}$ as a `UnitaryGate` (we've done the matrix exponential for you, see the provided code).
2. Prepare $|b\rangle = |1\rangle$ on the system qubit.
3. Implement the QPE step: Hadamards on the clock register, then controlled-$U$ and controlled-$U^2$, then inverse QFT.
4. Implement the controlled rotation on the ancilla: rotate by angle $2\arcsin(C/\lambda)$, conditioned on the clock register reading the binary pattern for that $\lambda$. (We've given you the two rotation angles; you supply the multi-controlled gates.)
5. Uncompute the QPE (undo steps in reverse).
6. Get the statevector, postselect on ancilla = 1 and clock = 00, and read off the (unnormalized) system-qubit amplitudes.
7. Compare your result to the classical solution `np.linalg.solve(A, b)`, normalized.

## Tip

Since there are only two possible eigenvalues here, the "controlled rotation" step doesn't need the general HHL machinery. You can implement it as two separate multi-controlled-RY gates, each conditioned on one specific clock bit pattern (`01` or `10`) via `X` gates to flip the condition, then `mcry`, then `X` again to undo.


In [ ]:
!pip install qiskit --quiet
!pip install qiskit[visualization] --quiet
!pip install qiskit_aer --quiet


In [ ]:
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate, UnitaryGate
from qiskit.quantum_info import Statevector

A = np.array([[1.5, -0.5], [-0.5, 1.5]])
b_vec = np.array([0, 1])

eigvals, eigvecs = np.linalg.eigh(A)
print("Eigenvalues of A:", eigvals)  # should be [1, 2]

t = np.pi / 2
U = expm(1j * A * t)      # e^{iAt}, controlled version used once (for lambda=1's bit)
U2 = U @ U                # U^2, controlled version used once (for lambda=2's bit)

print("U is unitary:", np.allclose(U @ U.conj().T, np.eye(2)))


In [ ]:
# Qubit indices
SYS, C0, C1, ANC = 0, 1, 2, 3

qc = QuantumCircuit(4)

# STUDENT TASK 1: prepare |b> = |1> on the system qubit
# YOUR CODE HERE

# STUDENT TASK 2: Hadamards on the clock register (C0, C1)
# YOUR CODE HERE

# STUDENT TASK 3: Implement the QPE step: apply controlled-$U$ with C0 as the control and SYS as the target, followed by controlled-$U^2$ with C1 as the control and SYS as the target.
# Hint: UnitaryGate(U).control(1) gives you a controlled version of U
# YOUR CODE HERE

# STUDENT TASK 4: inverse QFT on the clock register
# Hint: QFTGate(2).inverse(), applied to [C0, C1]
# YOUR CODE HERE

qc.draw('mpl')


In [ ]:
# STUDENT TASK 5: controlled rotation on the ancilla
C = 1.0  # normalization constant (choose C <= min eigenvalue, i.e. C <= 1)
theta_lambda1 = 2 * np.arcsin(C / 1)   # rotation angle when clock reads '01' (lambda=1)
theta_lambda2 = 2 * np.arcsin(C / 2)   # rotation angle when clock reads '10' (lambda=2)

# Hint: to condition on clock == '01' (i.e. C0=1, C1=0), temporarily X the C1 qubit
# so that both control qubits read 1, apply qc.mcry(theta, [C0, C1], ANC), then undo the X.
# Do the same (with a different X pattern) for clock == '10'.

# YOUR CODE HERE


In [ ]:
# STUDENT TASK 6: uncompute the QPE (apply the QPE steps in reverse:
# QFT (not inverse), controlled-U^2 inverse, controlled-U inverse, Hadamards)
# YOUR CODE HERE

qc.draw('mpl')


In [ ]:
# STUDENT TASK 7: extract and check the result
sv = Statevector.from_instruction(qc)

# bitstring order in Qiskit is (ANC, C1, C0, SYS) reading left to right in `sv.data` index
amp_sys0 = sv.data[int('1000', 2)]   # ANC=1, C1=0, C0=0, SYS=0
amp_sys1 = sv.data[int('1001', 2)]   # ANC=1, C1=0, C0=0, SYS=1

result_vec = np.array([amp_sys0, amp_sys1])
print("Unnormalized quantum solution:", result_vec)
print("Normalized quantum solution:  ", result_vec / np.linalg.norm(result_vec))

classical = np.linalg.solve(A, b_vec)
print("Normalized classical solution:", classical / np.linalg.norm(classical))

# how often does postselection on ancilla=1 succeed? (Probability of measuring ancilla = 1)
p_success = sum(v for k, v in sv.probabilities_dict().items() if k[0] == '1')
print(f"P(ancilla=1) success probability: {p_success:.3f}")


## Check your answer

Your normalized quantum solution should match the classical one to several decimal places: $[0.316, 0.949]$.

## Discussion

- Why did we need $C \le 1$ (the smaller eigenvalue) for the rotation constant? What would go wrong with a larger $C$? (Hint: think about what `arcsin` does when its argument exceeds 1.)
- Your postselection succeeds only some of the time (check the printed probability). This "postselection overhead" is one of the real practical costs of HHL that isn't captured by its headline exponential-speedup claim. Can you see why a *very* small success probability would eat into any speedup advantage?

## Extension

Try changing $\vec{b}$ to $|0\rangle$ (remove the initial `x` on the system qubit) and re-derive what the classical and quantum solutions should be. Does your circuit still get it right?


In [ ]:
# Extension: repeat with b = |0>
# YOUR CODE HERE
